In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import seaborn as sns

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/bootstrapped_fit")

schema = schema.post_index()

In [ ]:
asym_params = np.load(DATA_PATH / "ising_no_structure.npz")["params"][:, 8:].reshape(
    (-1, 8, 8)
)

_sym_params = np.load(DATA_PATH / "sym_ising_no_structure.npz")["params"][:, 8:]
sym_params = np.zeros_like(asym_params)
for i, p in enumerate(_sym_params):
    sym_params[i][np.triu_indices(8)] = p

asym_params[np.abs(asym_params) < 1e-2] = 0
sym_params[np.abs(sym_params) < 1e-2] = 0

In [ ]:
def in_degree(params, spin_idx: int) -> npt.NDArray[np.int64]:
    return (~np.isclose(np.delete(params[..., spin_idx], spin_idx, axis=-1), 0.0)).sum(
        axis=-1
    )


def out_degree(params, spin_idx: int) -> npt.NDArray[np.int64]:
    return (~np.isclose(np.delete(params[:, spin_idx], spin_idx, axis=-1), 0.0)).sum(
        axis=-1
    )


def in_strength(params, spin_idx: int) -> npt.NDArray[np.float64]:
    return np.abs(np.delete(params[..., spin_idx], spin_idx, axis=-1)).sum(axis=-1)


def out_strength(params, spin_idx: int) -> npt.NDArray[np.float64]:
    return np.abs(np.delete(params[:, spin_idx], spin_idx, axis=-1)).sum(axis=-1)


def degree(params, spin_idx: int) -> npt.NDArray[np.int64]:
    return in_degree(params, spin_idx) + out_degree(params, spin_idx)


def strength(params, spin_idx: int) -> npt.NDArray[np.float64]:
    return in_strength(params, spin_idx) + out_strength(params, spin_idx)

In [ ]:
sym_degree_ccw = degree(sym_params, 2)
sym_strength_ccw = strength(sym_params, 2)

asym_indegree_ccw = in_degree(asym_params, 2)
asym_outdegree_ccw = out_degree(asym_params, 2)
asym_instrength_ccw = in_strength(asym_params, 2)
asym_outstrength_ccw = out_strength(asym_params, 2)

sym_degree_pol = degree(sym_params, 5)
sym_strength_pol = strength(sym_params, 5)

asym_indegree_pol = in_degree(asym_params, 5)
asym_outdegree_pol = out_degree(asym_params, 5)
asym_instrength_pol = in_strength(asym_params, 5)
asym_outstrength_pol = out_strength(asym_params, 5)

In [ ]:
fig, axes = plt.subplots(ncols=2, constrained_layout=True)

axes[0].hist(sym_degree_ccw, label="Degree (symmetric)")
axes[0].hist(asym_indegree_ccw, label="In-degree (asymmetric)")
axes[0].hist(asym_outdegree_ccw, label="Out-degree (asymmetric)")

axes[1].hist(sym_strength_ccw, label="Strength (symmetric)")
axes[1].hist(asym_instrength_ccw, label="In-strength (asymmetric)")
axes[1].hist(asym_outstrength_ccw, label="Out-strength (asymmetric)")

axes[0].legend()
axes[1].legend()

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(5.77, 2), constrained_layout=True)

# sns.histplot(asym_outdegree_ccw - asym_indegree_ccw, ax=axes[0])
# sns.kdeplot(asym_outdegree_ccw - asym_indegree_ccw, ax=axes[0], warn_singular=False)
sns.distplot(asym_outstrength_ccw - asym_instrength_ccw, ax=axes[1])

print(np.percentile(asym_outstrength_ccw - asym_instrength_ccw, 5))

axes[0].set_title("Out-degree - in-degree")
axes[1].set_title("Out-strength - in-strength")

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(5.77, 2), constrained_layout=True)

sns.distplot(asym_outdegree_pol - asym_indegree_pol, ax=axes[0], kde=False)
sns.distplot(asym_outstrength_pol - asym_instrength_pol, ax=axes[1])

print(np.percentile(asym_outdegree_pol - asym_indegree_pol, 5))
print(np.percentile(asym_outstrength_pol - asym_instrength_pol, 5))

axes[0].set_title("Out-degree - in-degree")
axes[1].set_title("Out-strength - in-strength")

In [ ]:
asym_outdegree_pol

In [ ]:
asym_indegree_pol

In [ ]:
asym_params[0][:, 5]